In [1]:
!pip uninstall -y transformers tokenizers
!pip install transformers==4.44.2 tokenizers==0.19.1 datasets==2.19.0

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstallin

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import math
import torch
import random
import numpy as np
import gc
from datasets import load_dataset
from transformers import (
    RoFormerConfig, RoFormerForMaskedLM, RobertaTokenizerFast,
    Trainer, TrainingArguments, TrainerCallback
)

In [4]:
BASE_DIR = "/content/drive/MyDrive/AncientRusProject_RoFormer_V1"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer_BPE"
MODEL_DIR = f"{BASE_DIR}/mini_roformer_ancient_rus"
os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
print("⏳ Загрузка BPE токенизатора...")
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR, max_len=512)

⏳ Загрузка BPE токенизатора...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
special_tokens_dict = {
    'additional_special_tokens': [
        "[CTX_CHURCH]", "[CTX_DAILY]", "[CTX_LEGAL]",
        "[CTX_LIT]", "[CTX_EPIC]", "[CTX_SCIENCE]", "[UNK]"
    ]
}

In [7]:
tokenizer.add_special_tokens(special_tokens_dict)

7

In [8]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=False)

In [10]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (619 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [11]:
def group_texts(examples):
    block_size = 256
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated_examples[list(examples.keys())[0]]) // block_size) * block_size
    return {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

In [12]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [13]:
class PhysicalDegradationCollator:
    """
    Симулирует реальные повреждения исторических документов:
    1. Отломанные края (Edge Masking)
    2. Вытертые дыры (Span Masking)
    3. Стертые части слов (Random Subword Masking)
    """
    def __init__(self, tokenizer, mlm_prob=0.15, max_span=3, edge_prob=0.1):
        self.tokenizer = tokenizer
        self.mlm_prob = mlm_prob
        self.max_span = max_span
        self.edge_prob = edge_prob # Вероятность, что у предложения оторван край

    def __call__(self, features):
        input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        attention_mask = torch.tensor([f["attention_mask"] for f in features], dtype=torch.long)
        labels = input_ids.clone()

        batch_size, seq_len = input_ids.shape
        probability_matrix = torch.full(labels.shape, self.mlm_prob)

        # Защита спецтокенов
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
        ]
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

        # Базовая случайная маска
        masked_indices = torch.bernoulli(probability_matrix).bool()
        final_mask = masked_indices.clone()



        for i in range(batch_size):
            # 1. Edge Masking (Отломанный край)
            if random.random() < self.edge_prob:
                edge_len = random.randint(2, 5)
                is_start = random.choice([True, False])

                # Ищем границы, игнорируя <s> и </s> и теги контекста
                valid_indices = (~special_tokens_mask[i]).nonzero(as_tuple=True)[0]
                if len(valid_indices) > edge_len:
                    if is_start:
                        start_idx = valid_indices[0]
                        final_mask[i, start_idx : start_idx + edge_len] = True
                    else:
                        end_idx = valid_indices[-1]
                        final_mask[i, end_idx - edge_len + 1 : end_idx + 1] = True

            # 2. Span Masking (Вытертые дыры)
            for j in range(seq_len):
                if masked_indices[i, j]:
                    span_len = random.randint(1, self.max_span)
                    end_idx = min(j + span_len, seq_len)
                    if not special_tokens_mask[i, j:end_idx].any():
                        final_mask[i, j:end_idx] = True

        labels[~final_mask] = -100

        # Стандартные 80% [MASK], 10% рандом, 10% оригинал
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & final_mask
        input_ids[indices_replaced] = self.tokenizer.mask_token_id

        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & final_mask & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        input_ids[indices_random] = random_words[indices_random]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [14]:
data_collator = PhysicalDegradationCollator(tokenizer=tokenizer, mlm_prob=0.12, max_span=3, edge_prob=0.15)

In [15]:
config = RoFormerConfig(
    vocab_size=len(tokenizer),
    embedding_size=512,      # В RoFormer размер эмбеддинга задается отдельно
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=514,
    pad_token_id=tokenizer.pad_token_id,
    rotary_value=False       # Оптимизация для современных реализаций
)

In [16]:
model = RoFormerForMaskedLM(config)
model.resize_token_embeddings(len(tokenizer))
print(f"🧠 Параметры Mini-RoFormer: {model.num_parameters():,}")

🧠 Параметры Mini-RoFormer: 26,907,928


In [17]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.topk(logits, k=5, dim=-1).indices

In [18]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    return {
        "top1_accuracy": np.mean(preds[:, 0] == labels),
        "top3_accuracy": np.mean(np.any(preds[:, :3] == labels[:, None], axis=1)),
        "top5_accuracy": np.mean(np.any(preds[:, :5] == labels[:, None], axis=1)),
    }

In [19]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=15,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=False,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=True
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [21]:
trainer.train()

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,6.749500,6.559072,0.179614,0.238935,0.265711
800,5.764500,5.627226,0.224759,0.289846,0.319737
1200,5.158800,5.021106,0.259955,0.339998,0.377719
1600,4.703600,4.548697,0.298177,0.395627,0.439382
2000,4.335600,4.187491,0.337072,0.442599,0.488391
2400,4.052600,3.904926,0.374159,0.479827,0.524350
2800,3.823400,3.650824,0.405860,0.512539,0.556972
3200,3.643600,3.497361,0.429505,0.534198,0.576771
3600,3.510200,3.357950,0.450698,0.552768,0.593871
4000,3.396000,3.261255,0.463041,0.564403,0.605343


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=6120, training_loss=4.1922068552254075, metrics={'train_runtime': 4876.0618, 'train_samples_per_second': 160.787, 'train_steps_per_second': 1.255, 'total_flos': 2.308634890525901e+16, 'train_loss': 4.1922068552254075, 'epoch': 14.981640146878824})

In [22]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('/content/drive/MyDrive/AncientRusProject_RoFormer_V1/mini_roformer_ancient_rus/tokenizer_config.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V1/mini_roformer_ancient_rus/special_tokens_map.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V1/mini_roformer_ancient_rus/vocab.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V1/mini_roformer_ancient_rus/merges.txt',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V1/mini_roformer_ancient_rus/added_tokens.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V1/mini_roformer_ancient_rus/tokenizer.json')

In [23]:
print(f"\n📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
eval_results = trainer.evaluate()
print(f"Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
print(f"Top-5 Точность: {eval_results.get('eval_top5_accuracy', 0):.2%}")


📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:


Loss: 3.0636
Perplexity: 21.40
Top-5 Точность: 62.88%


In [24]:
from transformers import pipeline

In [25]:
roformer_pipe = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0 # Если GPU доступен, иначе ставь -1
)

# Хардкорные тесты (Обрати внимание: для RoFormer мы используем <mask >)
test_cases = [
    # 1. Классика: проверка падежей и логики (Летописи)
    {
        "desc": "📚 Летописи (на какую землю?)",
        "text": "[CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною.",
        "expected": "рускую / свою"
    },

    # 2. Судебник: проверка знания конкретных законов
    {
        "desc": "⚖️ Русская Правда (кого убили?)",
        "text": "[CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.",
        "expected": "мужь"
    },

    # 3. Бытовой: проверка понимания долгов
    {
        "desc": "🏡 Грамоты (про что пишут?)",
        "text": "[CTX_DAILY] поклоне ѿ ꙩндреꙗ · к ѥва · и к микифору про <mask> ѡкупи ꙩсподине",
        "expected": "серебро"
    },

    # 4. ТЕСТ НА ОТОРВАННЫЙ КРАЙ (Edge Masking)
    # У предложения нет начала, но RoPE должен понять, что к Василию шлют поклон
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванное начало",
        "text": "[CTX_DAILY] <mask> <mask> ко василью . а серебро ми отдай.",
        "expected": "поклонъ ѿ"
    },

    # 5. ТЕСТ НА ОТОРВАННЫЙ КОНЕЦ (Edge Masking)
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванный конец",
        "text": "[CTX_EPIC] Выезжал добрый <mask> из <mask> на <mask> <mask>",
        "expected": "молодец из города на добром коне"
    },

    # 6. ТЕСТ НА [UNK] (Реальная деградация из твоего датасета)
    # Проверяем, не сойдет ли модель с ума от тега [UNK]
    {
        "desc": "🧩 ТЕСТ [UNK]: Работа с нечитаемым текстом",
        "text": "[CTX_DAILY] [UNK] бь ѿ но [UNK] тию и св <mask> коуно",
        "expected": "Модель должна предложить варианты, игнорируя дыры [UNK]"
    },

    # 7. Церковный: проверка множественного числа
    {
        "desc": "⛪️ Церковный (кому сказал?)",
        "text": "[CTX_CHURCH] И рече господь къ <mask> своимъ, глаголя...",
        "expected": "ученикомъ / людемъ"
    }
]

print("\n" + "=" * 60)
print("🚀 СТАРТ ТЕСТИРОВАНИЯ ROFORMER (RoPE + DEGRADATION)")
print("=" * 60)

for idx, case in enumerate(test_cases, 1):
    print(f"\n[{idx}/7] {case['desc']}")
    print(f"📝 Текст: {case['text']}")
    print(f"🎯 Ожидалось (смысл): {case['expected']}")

    # Если масок несколько, пайплайн вернет список списков
    mask_count = case['text'].count("<mask>")
    results = roformer_pipe(case['text'], top_k=3)

    # Нормализуем вывод (если маска одна, оборачиваем в список для единообразия)
    if mask_count == 1:
        results = [results]

    for i, mask_res in enumerate(results):
        print(f"  ➡️ Маска {i+1}: ", end="")
        preds = []
        for res in mask_res:
            # Очищаем от байтового пробела RoBERTa/RoFormer
            clean_word = res['token_str'].replace("Ġ", "").strip()
            score = res['score'] * 100
            preds.append(f"'{clean_word}' ({score:.1f}%)")
        print(" | ".join(preds))


🚀 СТАРТ ТЕСТИРОВАНИЯ ROFORMER (RoPE + DEGRADATION)

[1/7] 📚 Летописи (на какую землю?)
📝 Текст: [CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною.
🎯 Ожидалось (смысл): рускую / свою
  ➡️ Маска 1: 'свою' (81.5%) | 'рускую' (3.2%) | 'правую' (1.8%)

[2/7] ⚖️ Русская Правда (кого убили?)
📝 Текст: [CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.
🎯 Ожидалось (смысл): мужь
  ➡️ Маска 1: 'мужь' (16.7%) | 'тать' (8.7%) | 'мужа' (5.8%)

[3/7] 🏡 Грамоты (про что пишут?)
📝 Текст: [CTX_DAILY] поклоне ѿ ꙩндреꙗ · к ѥва · и к микифору про <mask> ѡкупи ꙩсподине
🎯 Ожидалось (смысл): серебро
  ➡️ Маска 1: '·' (98.8%) | 'серебро' (0.3%) | 'твоѥ' (0.1%)

[4/7] 🧨 ТЕСТ ROPE: Оторванное начало
📝 Текст: [CTX_DAILY] <mask> <mask> ко василью . а серебро ми отдай.
🎯 Ожидалось (смысл): поклонъ ѿ
  ➡️ Маска 1: 'От' (46.0%) | 'Грамота' (7.0%) | 'Покланяние' (5.8%)
  ➡️ Маска 2: 'ѧкима' (9.7%) | 'аку' (8.4%) | 'игната' (7.2%)

[5/7] 🧨 ТЕСТ ROPE: Оторванный конец
📝 Текст: [CTX_EPIC] Вы